# Классификация: логистическая регрессия и метрики

## Содержание

0. Подготовка среды и подключение Weights & Biases
1. Данные: отток клиентов (Telco Customer Churn)
2. Логистическая регрессия
3. Метрики классификации
4. Порог: стоимость ошибок и пропускная способность
5. Практикум

Это занятие продолжает второе. Там вы обучали линейную регрессию и предсказывали число; теперь **та же линейная комбинация признаков** превращается в вероятность через сигмоиду, а целевая переменная становится классом. Пайплайн прежний (данные → EDA → модель → метрики → интерпретация), но метрики классификации совсем другие — им посвящена большая часть занятия.

Демонстрационный пример — задача прогноза **оттока клиентов** телеком-оператора (датасет Telco Customer Churn). Практикум (раздел 5) вы выполняете на **датасете своего проекта из занятия 2**. Запуски логируем в Weights & Biases (регистрация — из task 01). Выводы после каждого блока записывайте **в ячейке markdown прямо в блокноте**.

## 0. Подготовка среды и подключение Weights & Biases

К библиотекам первых двух занятий добавляется `wandb`. Датасет скачивается из интернета (небольшой файл), поэтому первый запуск займёт немного больше времени.

Для входа в W&B нужен API-ключ: https://wandb.ai/authorize (регистрация — из task 01). В Google Colab ключ удобно хранить в «Секретах» (значок ключа слева) и не вписывать в код.

**Примечание.** Если с W&B технические проблемы (нет сети или аккаунта), замените `wandb.init(...)` на `wandb.init(mode="disabled")` — все вызовы станут холостыми, а код не сломается.

In [1]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)

import wandb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42

# Логин в W&B (регистрация — из task 01). Ключ: https://wandb.ai/authorize
wandb.login()

ModuleNotFoundError: No module named 'wandb'

## 1. Данные: отток клиентов (Telco Customer Churn)

Датасет описывает клиентов телеком-оператора: услуги, тарифы, срок обслуживания и факт ухода. Это классическая задача **бинарной классификации с учителем**.

**Постановка.** Объект — клиент. Момент предсказания — до начала кампании удержания. Действие после score — предложить скидку, позвонить, ничего не делать. Модель не заменяет решение: между вероятностью и действием стоят порог, стоимость ошибок и пропускная способность отдела удержания.

| Поле | Тип | Смысл |
|---|---|---|
| customerID | идентификатор | в признаки не входит |
| gender, Partner, Dependents | категориальный | демография |
| tenure | числовой | сколько месяцев с нами |
| PhoneService, MultipleLines | категориальный | телефония |
| InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport | категориальный | интернет-услуги |
| StreamingTV, StreamingMovies | категориальный | стриминг |
| Contract, PaperlessBilling, PaymentMethod | категориальный | договор и оплата |
| MonthlyCharges | числовой | месячный платёж |
| TotalCharges | числовой | суммарные платежи (в файле — текст с пустыми значениями) |
| Churn | целевая | Yes — клиент ушёл, No — остался |

Формально: объект $x \in \mathbb{R}^d$ (числовые признаки и one-hot категориальные), целевая переменная $y \in \{0, 1\}$; ищем $f(x) \to [0, 1]$ — вероятность оттока.

**Важная оговорка.** Доля оттока в датасете около 26% — это учебное распределение. В разрезе сегментов и периодов она отличается, а precision напрямую зависит от base rate: при меньшей доле уходящих precision при том же recall будет ниже.

In [ ]:
DATA_URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
            "master/data/Telco-Customer-Churn.csv")
LOCAL_PATH = "telco_churn.csv"

# Скачиваем один раз и кэшируем рядом с блокнотом, чтобы не качать при каждом запуске
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
else:
    df = pd.read_csv(DATA_URL)
    df.to_csv(LOCAL_PATH, index=False)

print("Размер таблицы:", df.shape)
print("Доля оттока:", round((df['Churn'] == 'Yes').mean(), 3))
df.head()

In [ ]:
# Аудит: целевая переменная, пропуски, дубликаты, кардинальность
print("Целевая переменная:")
print(df['Churn'].value_counts())
print("\nДубликаты строк:", df.duplicated().sum())

# TotalCharges записан как текст, часть значений пустая
print("\nTotalCharges — тип:", df['TotalCharges'].dtype,
      "| некорректных значений:", int(pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum()))

# Готовим рабочую таблицу: убираем идентификатор, приводим типы
d = df.drop(columns=['customerID']).copy()
d['TotalCharges'] = pd.to_numeric(d['TotalCharges'], errors='coerce')
d['Churn'] = (d['Churn'] == 'Yes').astype(int)

num = [c for c in d.select_dtypes(include=np.number).columns if c != 'Churn']
cat = [c for c in d.columns if c not in num + ['Churn']]

print("\nЧисловых признаков:", len(num), "| категориальных:", len(cat))
print("Кардинальность категориальных:", {c: d[c].nunique() for c in cat})

### Контроль утечек (leakage checklist)

- Удалить `customerID` — это идентификатор, а не признак.
- Исключить признаки, доступные только **после** целевого события (обращения в поддержку, причина расторжения, финальный счёт).
- Не обучать scaler/encoder/imputer до разбиения — весь препроцессинг держать внутри `Pipeline`.
- Осторожно с признаками, рассчитанными с использованием будущего (агрегаты за период после момента предсказания).
- Учитывать, что случайный split может быть оптимистичным, если у одного клиента в данных несколько строк.

**Задача 1.**
Проведите аудит датасета (пропуски, дубликаты, типы, кардинальность) и назовите минимум **четыре** потенциальных источника утечки именно для этой задачи, объяснив, чем каждый опасен. Ответ — в ячейке ниже.

**Ваш ответ:**

_..._

## 2. Логистическая регрессия

### Математический аппарат

Линейная комбинация признаков — та же, что и в занятии 2, но пропущенная через **сигмоиду**, чтобы результат лежал в $[0, 1]$ и трактовался как вероятность:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad P(y=1 \mid x) = \hat{y} = \sigma(w^\top x + b)$$

Эквивалентная запись через log-odds: $\log\frac{\hat{y}}{1-\hat{y}} = w^\top x + b$. После стандартизации знак коэффициента показывает направление связи, а $\exp(w_j)$ — во сколько раз меняются шансы оттока при росте признака на одно стандартное отклонение (при прочих равных).

Веса подбираются максимизацией правдоподобия, что даёт **binary cross-entropy**:

$$L(w) = -\frac{1}{n}\sum_{i=1}^{n}\left[\,y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\,\right] \to \min_{w}$$

*Смысл:* за уверенную ошибку ($\hat{y} \to 0$ при $y=1$) штраф растёт до бесконечности — модель наказывается именно за неверную уверенность, а не за сам неверный класс.

Градиент совпадает по виду с градиентом MSE из занятия 2 — «ошибка, умноженная на признак» (вывод — задача 2):

$$\frac{\partial L}{\partial w} = \frac{1}{n}X^\top(\hat{y} - y)$$

**Регуляризация.** Штраф $\lambda\|w\|^2$ (L2) — не произвольная добавка: он получается как MAP-оценка при гауссовском априоре на веса. В `sklearn` сила регуляризации задаётся наоборот: `C = 1/λ`, поэтому меньше `C` — сильнее сжатие весов. Подробный разбор регуляризации — в следующих занятиях.

**Интерпретация коэффициентов (параллель с занятием 2).** В линейной регрессии $w_j$ — изменение прогноза при росте $x_j$; здесь $w_j$ — изменение логарифма отношения шансов. Предостережение про мультиколлинеарность из P.S. второго занятия справедливо и здесь.

In [ ]:
X = d[num + cat]
y = d['Churn']

# 60/20/20 со стратификацией: validation — для выбора модели и порога, test — только финал
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE
)

# Весь препроцессинг — внутри Pipeline, чтобы статистики не «утекали» из test
preproc = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat),
])

model = Pipeline([
    ('prep', preproc),
    ('clf', LogisticRegression(C=0.3, max_iter=1500, solver='liblinear')),
])
model.fit(X_train, y_train)

p_val = model.predict_proba(X_val)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]

print("Обучающая:", X_train.shape, "| валидация:", X_val.shape, "| тест:", X_test.shape)
print("Признаков после кодирования:", len(model.named_steps['prep'].get_feature_names_out()))

**Примечание 1.** В признаки идут только `num + cat`; `Churn` — цель, `customerID` исключён как идентификатор. Один и тот же `Pipeline` применяется к train/val/test, поэтому статистики импутера и скейлера считаются только по train.

**Примечание 2.** Порог и сравнение моделей делаем на validation; test открываем один раз в разделе 4. Это защищает от подгонки под тест.

**Примечание 3.** Конфигурацию запуска (модель, `C`, схему split) и метрики логируем в W&B.

**Задача 2 (на вывод).**
Выведите формулу градиента binary cross-entropy, приведённую выше: возьмите производную $L$ по $w$ для одного объекта и используйте свойство сигмоиды $\sigma'(z) = \sigma(z)\,(1-\sigma(z))$. Оформите вывод формулами в ячейке markdown.

**Ваш вывод:**

_..._

## 3. Метрики классификации

### Математический аппарат

Предсказания раскладываются в **матрицу ошибок**:

| | предсказано 0 | предсказано 1 |
|---|---|---|
| **на самом деле 0** | TN | FP (ложная тревога) |
| **на самом деле 1** | FN (пропуск оттока) | TP |

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

*Осторожно:* при доле оттока 26% классификатор «всегда останется» уже даёт accuracy 0.74 — формально неплохо, но бесполезно. Поэтому нужны метрики, чувствительные к классам:

$$Precision = \frac{TP}{TP + FP} = P(y=1 \mid \hat{y}=1), \qquad Recall = \frac{TP}{TP + FN} = P(\hat{y}=1 \mid y=1)$$

$$F_1 = \frac{2 \cdot Precision \cdot Recall}{Precision + Recall}$$

*Смысл:* precision — какой доле «предупредили — уйдёт» можно верить; recall — какую долю реальных уходов мы поймали. F1 — гармоническое среднее, поэтому резко падает, если одна из метрик близка к нулю.

**ROC-кривая** строится по всем порогам: $TPR(t)$ против $FPR(t)$. Площадь под ней — вероятность того, что случайный ушедший клиент получит больший скор, чем случайный оставшийся:

$$AUC = \int_0^1 TPR\,dFPR = P\big(s(x^+) > s(x^-)\big)$$

**PR-кривая** (precision против recall) чувствительнее при редком позитивном классе: её случайный уровень — доля позитивов $\pi$, а не диагональ. Площадь под PR-кривой оценивают метрикой **Average Precision (AP)**.

**Log-loss** (та же binary cross-entropy) измеряет качество самих вероятностей и сильно штрафует уверенные ошибки.

In [ ]:
# Наивный бейзлайн для контекста: всегда предсказывает самый частый класс
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"Dummy accuracy на test: {accuracy_score(y_test, dummy.predict(X_test)):.3f}")
print(f"Доля оттока на test: {y_test.mean():.3f}")

# Метрики при нейтральном пороге 0.5
pred_05 = (p_test >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_05)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion matrix:")
print(cm)
print(f"Precision: {tp / (tp + fp):.3f}")
print(f"Recall:    {tp / (tp + fn):.3f}")
print(f"F1:        {2 * tp / (2 * tp + fp + fn):.3f}")
print()
print(classification_report(y_test, pred_05, digits=3))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_test)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, p_test)
ap = average_precision_score(y_test, p_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, p_test):.3f}")
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label="Случайный классификатор")
axes[0].set_xlabel("FPR (доля ложных тревог)")
axes[0].set_ylabel("TPR (recall)")
axes[0].set_title("ROC-кривая")
axes[0].legend()

axes[1].plot(rec_curve, prec_curve, label=f"Average Precision = {ap:.3f}")
axes[1].axhline(y_test.mean(), linestyle='--', color='gray',
                label=f"Доля оттока = {y_test.mean():.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("PR-кривая")
axes[1].legend()

plt.tight_layout()
plt.show()

**Примечание.** Доля оттока в датасете 26% завышает precision относительно многих реальных сегментов: при меньшей доле уходящих precision при том же recall падает. Высокий ROC-AUC не переносится на другой сегмент автоматически — base rate нужно учитывать.

**Задача 3.**
Сравните precision, recall и F1 при пороге 0.5 с ожидаемым поведением при доле оттока 5% (вместо 26%): как изменится precision при том же recall? Объясните через формулу precision и через уровень PR-кривой. Ответ — в «Ваш ответ».

**Ваш ответ:**

_..._

**Задача со звёздочкой.**
Докажите равенство $AUC = P\big(s(x^+) > s(x^-)\big)$. Подсказка: запишите AUC как интеграл по всем порогам $t$ и сверните его к ранговой статистике Манна–Уитни (доле пар «ушёл–остался», где ушедший получил больший скор).

**Ваше доказательство:**

_..._

## 4. Порог: стоимость ошибок и пропускная способность

### Математический аппарат

Порог 0.5 — просто нейтральное значение, а не «правильное». Пусть ложная тревога стоит $c_{FP}$ (неоправданная скидка, лишний звонок), а пропуск оттока — $c_{FN}$ (потерянный клиент). Ожидаемая цена решения при пороге $t$:

$$C(t) = c_{FP} \cdot FP(t) + c_{FN} \cdot FN(t), \qquad t^* = \arg\min_{t} C(t)$$

Для откалиброванной модели есть ориентир: предлагать удержание выгодно, когда ожидаемая цена ложной тревоги ниже цены пропуска:

$$(1-p)\,c_{FP} < p\,c_{FN} \quad\Longleftrightarrow\quad p > \frac{c_{FP}}{c_{FP} + c_{FN}}$$

*Смысл:* чем дороже потерянный клиент относительно лишнего удержания, тем ниже порог. При $c_{FN} = 20\,c_{FP}$ получаем $t^* \approx 0.05$ — звонить стоит уже при 5% вероятности ухода.

**Ограничение по пропускной способности.** Даже оптимальный по цене порог может породить больше касаний, чем отдел удержания успевает обработать. Тогда порог выбирают как минимум стоимости при ограничении: число предложений на 10 000 клиентов не больше $K$.

In [ ]:
C_FP, C_FN = 1, 20  # потерянный клиент условно «дороже» лишнего удержания в 20 раз

thresholds = np.linspace(0.01, 0.99, 99)
rows = []
for t in thresholds:
    tn, fp, fn, tp = confusion_matrix(y_val, p_val >= t).ravel()
    rows.append((t, fp + C_FN * fn, (p_val >= t).mean() * 10000))

th_table = pd.DataFrame(rows, columns=['threshold', 'cost', 'offers_per_10k'])

best = th_table.loc[th_table['cost'].idxmin()]
print(f"Порог по стоимости: {best['threshold']:.2f} "
      f"(cost = {best['cost']:.0f}, предложений/10k = {best['offers_per_10k']:.0f})")
print(f"Теоретический ориентир c_FP/(c_FP+c_FN) = {C_FP / (C_FP + C_FN):.3f}")

cap = th_table[th_table['offers_per_10k'] <= 500]
if len(cap) == 0:
    print(f"Ограничение <= 500 предложений/10k недостижимо: минимум "
          f"{th_table['offers_per_10k'].min():.0f} при пороге 0.99.")
else:
    best_cap = cap.loc[cap['cost'].idxmin()]
    print(f"С ограничением <= 500 предложений/10k: порог {best_cap['threshold']:.2f} "
          f"(cost = {best_cap['cost']:.0f})")

plt.figure(figsize=(8, 5))
plt.plot(th_table['threshold'], th_table['cost'], label='Стоимость FP + 20·FN')
plt.axvline(best['threshold'], linestyle='--', color='red',
            label=f"порог по стоимости = {best['threshold']:.2f}")
plt.axvline(0.5, linestyle=':', color='gray', label='Порог 0.5')
plt.xlabel('Порог')
plt.ylabel('Стоимость')
plt.title('Выбор порога по стоимости ошибок')
plt.legend()
plt.show()

T_STAR = best['threshold']

In [ ]:
# Финальная оценка на test — один раз, при пороге, выбранном на validation
pred_star = (p_test >= T_STAR).astype(int)
cm_star = confusion_matrix(y_test, pred_star)

result = {
    'threshold': round(float(T_STAR), 2),
    'accuracy': accuracy_score(y_test, pred_star),
    'precision': precision_score(y_test, pred_star),
    'recall': recall_score(y_test, pred_star),
    'f1': f1_score(y_test, pred_star),
    'roc_auc': roc_auc_score(y_test, p_test),
    'average_precision': average_precision_score(y_test, p_test),
    'log_loss': log_loss(y_test, p_test),
}
print("Confusion matrix (порог из validation):")
print(cm_star)
for k, v in result.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
print(f"Предложений на 10 000 клиентов: {pred_star.mean() * 10000:.0f}")

run = wandb.init(
    project="apml-course",
    name="logreg-churn-baseline",
    config={"model": "LogisticRegression", "C": 0.3, "split": "60/20/20",
            "cost_ratio_FN_FP": C_FN, "threshold": float(T_STAR)},
)
wandb.log(result)
run.finish()

**Практическая применимость.** Низкий порог повышает recall почти до единицы, но создаёт много ложных тревог: дёргаем клиентов, которые и так остались бы. Качество модели и операционная пригодность — разные вещи: порог выбирают вместе с владельцем процесса удержания, зная стоимость ошибок и сменную пропускную способность.

**Задача 4.**
Исследуйте отношения $c_{FN}/c_{FP} \in \{1, 5, 20, 100\}$: для каждого подберите порог на validation, затем посчитайте на test recall, precision, FP, FN и стоимость. Добавьте ограничение по нагрузке и сравните пороги. Сформулируйте рекомендацию для двух сценариев: массовая рассылка со скидкой и персональный звонок менеджера. Ответ — в «Ваш ответ».

**Ваш ответ:**

_..._

## 5. Практикум

Как и на втором занятии, все задания выполняются на **датасете вашего проекта** (том же, что в занятии 2). Если целевая переменная непрерывная (регрессионная задача) — сформулируйте бинарную подзадачу, например «выше медианы / ниже медианы», и отметьте, насколько такая постановка искусственна для вашей предметной области.

Каждый запуск модели логируйте в W&B отдельным run. После каждого задания заполните ячейку **«Вывод»**.

### Задание 5.1. Постановка и проверка утечек

Продумайте leakage checklist для своего датасета: есть ли признаки, недоступные в момент предсказания, и признаки, вычисленные с использованием будущего. Сформулируйте: что является «позитивом» (класс 1); какова его доля; какая цена у FP и FN в вашей предметной области и что страшнее.

**Ваш ответ:**

_..._

### Задание 5.2. Базовый классификатор

Обучите логистическую регрессию на своих данных (стратифицированный split, весь препроцессинг внутри `Pipeline`). Сравните с `DummyClassifier`. Посчитайте accuracy, precision, recall, F1, ROC-AUC, AP, log-loss и залогируйте запуск в W&B. В «Выводе»: какая метрика оказалась неожиданной и почему.

**Вывод:**

_..._

### Задание 5.3. Матрица ошибок

Выведите `confusion_matrix` своей модели и опишите, какие ошибки она делает чаще (FP или FN) и насколько это плохо для вашей задачи из 5.1.

**Вывод:**

_..._

### Задание 5.4. Порог: стоимость и пропускная способность

Задайте $c_{FP}$ и $c_{FN}$ из обоснования 5.1. Подберите порог **только на validation** (перебор из раздела 4) и оцените метрики на test. Добавьте ограничение по нагрузке: не более K предложений на 10 000 объектов, $K \in \{50, 100, 500\}$. Сравните пороги 0.5, по максимуму F1 и по минимуму стоимости.

**Вывод:**

_..._

### Задание 5.5. Взвешивание классов

Обучите вторую модель — ту же логистическую регрессию с `class_weight='balanced'`. Сравните precision/recall/F1 с базовой, залогируйте запуск в W&B и сравните два run в интерфейсе. В «Выводе»: помогло ли взвешивание именно по вашей главной метрике из 5.1 и как оно повлияло на вероятности модели.

**Вывод:**

_..._

### Задание 5.6. Итоговое резюме

В 5–7 предложениях: какая метрика будет главной в вашем проекте и почему; как повели себя ROC-AUC и AP на ваших данных; какой порог и какую модель вы зафиксируете как классификационный бейзлайн; какие ограничения (base rate, leakage, drift) нужно указать в отчёте.

**Резюме:**

_..._

---

## P.S. Калибровка вероятностей

**Калибровка** — согласие между предсказанной вероятностью и реальной частотой: среди клиентов с $\hat{y} \approx 0.8$ уходит примерно 80%. Логистическая регрессия обычно откалибрована неплохо, а бустинг и нейросети — часто нет.

Проверяется **калибровочной кривой** (reliability diagram): объекты разбиваются на бины по $\hat{y}$, и средняя предсказанная вероятность сравнивается с эмпирической долей позитивов. Идеал — диагональ. Исправляют постфактум через **Platt scaling** или **изотоническую регрессию** (`CalibratedClassifierCV`).

**Когда это важно.** Когда сама вероятность — продукт: кредитный скоринг, медицина, оценка ожидаемой выручки от кампании удержания. Именно для откалиброванной модели справедлива формула порога $t^* = c_{FP}/(c_{FP}+c_{FN})$ из раздела 4. Если нужен только ранжинг или класс по порогу, достаточно ROC-AUC и AP.

**Замечание про `class_weight`.** Взвешивание классов меняет обучение и почти всегда портит калибровку: модель начинает систематически завышать вероятности редкого класса. Поэтому взвешивание и сдвиг порога не стоит применять одновременно — это двойной учёт дисбаланса.

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, p_test, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker='o', label='Логистическая регрессия')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Идеальная калибровка')
plt.xlabel("Средняя предсказанная вероятность")
plt.ylabel("Доля оттока в бине")
plt.title("Калибровочная кривая")
plt.legend()
plt.show()

### Справочные материалы

- Telco Customer Churn (IBM): https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
- Документация scikit-learn (LogisticRegression): https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- Метрики классификации в scikit-learn: https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics
- Документация Pandas: https://pandas.pydata.org/docs/